# Lösung: Vortrainiertes YOLO anwenden

1. Wenden Sie YOLO26n auf `strasse.jpg` und `bus.jpg` an und zählen Sie die Objekte je Klasse.
2. Vergleichen Sie YOLO26n und YOLO26s hinsichtlich Anzahl der Detektionen, mittlerer Confidence und Laufzeit.
3. Untersuchen Sie, wie die Anzahl erkannter Personen vom Schwellenwert `conf` abhängt.
4. Wenden Sie das vortrainierte Modell auf `testbild.jpg` an und erklären Sie das Ergebnis.

## Umgebung

Die Notebooks laufen lokal oder in Google Colab. Für Colab `use_colab = True` setzen und unter *Laufzeit > Laufzeittyp ändern* eine GPU wählen.

Die Version von `ultralytics` ist festgelegt, damit die Notebooks auch in späteren Semestern laufen.

In [ ]:
use_colab = False

if use_colab:
    %pip install -q ultralytics==8.4.215
    from google.colab import drive
    drive.mount("/content/drive")
    base_path = "/content/drive/MyDrive"
else:
    # lokal einmalig installieren: pip install ultralytics==8.4.215
    base_path = ".."

import ultralytics
ultralytics.checks()

In [ ]:
import cv2
import matplotlib.pyplot as plt


def show_image(image_bgr, title="", size=(10, 7)):
    plt.figure(figsize=size)
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

## 1. Objekte zählen

In [ ]:
import os
from collections import Counter
from ultralytics import YOLO
from ultralytics.utils import ASSETS

model_n = YOLO("yolo26n.pt")
images = [str(ASSETS / "bus.jpg"), "strasse.jpg"]     # strasse.jpg liegt im Ordner des Notebooks
for b in images:
    if not os.path.exists(b):
        print("Datei nicht gefunden:", b)
images = [b for b in images if os.path.exists(b)]

for image in images:
    r = model_n.predict(image, verbose=False)[0]
    print(os.path.basename(image), dict(Counter(r.names[int(c)] for c in r.boxes.cls)))
    show_image(r.plot(), os.path.basename(image))

Vergleichen Sie die gefundenen Klassen mit den 80 COCO-Klassen: Erkannt werden nur Objekte aus diesen Klassen, auf einem Straßenbild z. B. Autos, Personen oder Ampeln. Gebäude, Bäume oder Straßenschilder ohne eigene Klasse findet das Modell nicht. Es kann nur erkennen, was es gelernt hat.

## 2. YOLO26n gegen YOLO26s

In [ ]:
import pandas as pd

model_s = YOLO("yolo26s.pt")
rows = []
for name, m in [("YOLO26n", model_n), ("YOLO26s", model_s)]:
    for image in images:
        m.predict(image, verbose=False)              # Aufwärmen
        r = m.predict(image, verbose=False)[0]
        rows.append({"Modell": name, "image": os.path.basename(image),
                       "Detektionen": len(r.boxes),
                       "mittlere Confidence": float(r.boxes.conf.mean()) if len(r.boxes) else 0,
                       "Inferenz in ms": r.speed["inference"]})
pd.DataFrame(rows).round(2)

Das größere Modell findet meist etwas mehr Objekte mit höherer Confidence, benötigt dafür aber ein Mehrfaches an Rechenzeit.

## 3. Personen in Abhängigkeit von `conf`

In [ ]:
import numpy as np

image = str(ASSETS / "bus.jpg")
confs = np.arange(0.05, 1.0, 0.05)
persons = []
for conf in confs:
    r = model_n.predict(image, conf=conf, classes=[0], verbose=False)[0]   # Klasse 0 = person
    persons.append(len(r.boxes))

plt.step(confs, persons, where="post")
plt.xlabel("conf")
plt.ylabel("erkannte Personen")
plt.grid(alpha=0.3)
plt.show()

## 4. Das Testbild

`testbild.jpg` liegt im Ordner des Notebooks. Darauf liegen Euro-Münzen zwischen Schere, Stiften und anderen Gegenständen.

In [ ]:
for conf in [0.25, 0.05]:
    r = model_n.predict("testbild.jpg", conf=conf, verbose=False)[0]
    print(f"conf={conf}:", [f"{r.names[int(c)]} {p:.2f}" for c, p in zip(r.boxes.cls, r.boxes.conf)])
show_image(r.plot(), "YOLO26n (COCO) auf testbild.jpg")